<a href="https://colab.research.google.com/github/AaravGang/QCMate/blob/main/QCMate_fcf_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 119.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:

from roboflow import Roboflow
from google.colab import userdata
rf = Roboflow(api_key=userdata.get('ROBOFLOW'))
project = rf.workspace("aarav-gang").project("ayushannotations-d36ii")
version = project.version(5)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to AyushAnnotations-5 in yolov8:: 100%|██████████| 2011/2011 [00:00<00:00, 3774.95it/s]


In [7]:
import os
import shutil

gdrive_project_dir = "/content/drive/MyDrive/QCMate"
os.makedirs(gdrive_project_dir, exist_ok=True)

dataset_local_path = dataset.location
zip_destination = f"{gdrive_project_dir}/FCF_DATASET_1024"

print("Zipping and backing up dataset to Google Drive...")
shutil.make_archive(zip_destination, 'zip', dataset_local_path)
print(f"🎉 Dataset safely backed up as a zip file in: {gdrive_project_dir}/FCF_DATASET_1024.zip")

Zipping and backing up dataset to Google Drive...
🎉 Dataset safely backed up as a zip file in: /content/drive/MyDrive/QCMate/FCF_DATASET_1024.zip


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
print(dataset.location)

# 100 epochs
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=1024,
    batch=8,
    box=12.0,
    device=0,
    plots=True
)

/content/AyushAnnotations-5
Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=12.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/AyushAnnotations-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimi

In [ ]:
import glob
from IPython.display import Image, display

# Fetch test images
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")

if test_images:
    test_image = test_images[0]

    # Run prediction
    predict_results = model.predict(source=test_image, save=True, conf=0.25)

    # Render the result image
    output_path = f"{predict_results[0].save_dir}/{test_image.split('/')[-1]}"
    display(Image(filename=output_path))
else:
    print("No test images found. Check your dataset configuration.")

In [ ]:
onnx_path = model.export(format="onnx")

weights_dir = "/content/runs/detect/train-2/weights"
gdrive_weights_dir = "/content/drive/MyDrive/QCMate/1024_version2/weights"
os.makedirs(gdrive_weights_dir, exist_ok=True)

try:
    shutil.copy(f"{weights_dir}/best.pt", f"{gdrive_weights_dir}/best.pt")
    shutil.copy(f"{weights_dir}/best.onnx", f"{gdrive_weights_dir}/best.onnx")
    print(f"Success! Weights backed up to Google Drive at: {gdrive_weights_dir}/")
except FileNotFoundError:
    print("Weights files not found. Ensure your training and export loops completed successfully.")